# `puccinialin` smoke test

In [ ]:
import inspect
import puccinialin
from puccinialin import get_triple, setup_rust

public = [n for n in dir(puccinialin) if not n.startswith("_") and n != "annotations"]
print("public names:", public)
assert "setup_rust" in public and "get_triple" in public

sig = inspect.signature(setup_rust)
assert list(sig.parameters) == ["installation_dir", "program", "file"]
assert sig.parameters["installation_dir"].default is None
assert sig.parameters["program"].default is None
assert sig.return_annotation == "dict[str, str]"
print("OK — setup_rust(installation_dir=None, program=None, file=stderr) -> dict[str, str]")

In [ ]:
import io, platform
from puccinialin import get_triple
from puccinialin._target import rustup_targets

diag = io.StringIO()
triple = get_triple(diag)
print("triple:", triple)
print(diag.getvalue().strip())
assert triple in rustup_targets, f"{triple} not a known rustup target"
assert "SOABI" in diag.getvalue() or "platform" in diag.getvalue()
# platform.machine() spells some architectures differently from rust triples (arm64, AMD64, armv7l).
machine = {"arm64": "aarch64", "ARM64": "aarch64", "AMD64": "x86_64"}.get(platform.machine(), platform.machine())
arch = triple.split("-")[0]
assert arch == machine or machine.startswith(arch), (triple, platform.machine())
print("OK — triple matches the machine arch and the rustup registry")

In [ ]:
import io, os, platform
from puccinialin._target import Target

target = Target(io.StringIO())
assert target.system == platform.system()
assert target.is_windows() is (platform.system() == "Windows")
assert target.exe_suffix() == (".exe" if target.is_windows() else "")
assert target.env_path_separator() == os.pathsep
print(f"system={target.system} exe_suffix={target.exe_suffix()!r} path_sep={target.env_path_separator()!r}")
print("OK — Target's platform answers agree with the os/platform stdlib")

In [ ]:
from puccinialin._target import rustup_targets

assert len(rustup_targets) == len(set(rustup_targets)), "duplicate triples"
for triple in rustup_targets:
    parts = triple.split("-")
    assert 2 <= len(parts) <= 4, triple
    assert triple == triple.lower(), triple
families = {t.split("-")[0] for t in rustup_targets}
assert {"aarch64", "x86_64"} <= families
print(f"{len(rustup_targets)} registered triples across {len(families)} arch families")
print("OK — rustup target registry is well-formed")